In [1]:
import re

def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)
    return " ".join(text.split())


def f1_score(pred, gold):
    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()

    common = set(pred_tokens) & set(gold_tokens)

    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

In [ ]:
def predict_topk(question, context, k=5):
    inp = f"question: {question} context: {context}"

    inputs = tokenizer(inp, return_tensors="pt").to(model.device)

    outputs = model.generate(
        inputs["input_ids"],
        max_length=32,
        num_beams=k,
        num_return_sequences=k,
        early_stopping=True
    )

    preds = [
        tokenizer.decode(o, skip_special_tokens=True)
        for o in outputs
    ]

    return preds

In [ ]:
def score_candidate(pred, question):
    pred_tokens = set(normalize(pred).split())
    q_tokens = set(normalize(question).split())

    return len(pred_tokens & q_tokens)


def rerank_answers(question, candidates):
    scored = [(c, score_candidate(c, question)) for c in candidates]
    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    return scored[0][0]

In [ ]:
from tqdm import tqdm

preds_rerank = []
all_candidates = []

for q, ctx in tqdm(zip(test_q, test_ctx), total=len(test_q)):
    candidates = predict_topk(q, ctx, k=5)
    best = rerank_answers(q, candidates)

    preds_rerank.append(best)
    all_candidates.append(candidates)

In [ ]:
hit1 = 0
hit5 = 0
f1_total = 0

for p, g, candidates in zip(preds_rerank, test_a, all_candidates):

    # Hit@1
    if normalize(p) == normalize(g):
        hit1 += 1

    # Hit@5
    if any(normalize(c) == normalize(g) for c in candidates):
        hit5 += 1

    # F1
    f1_total += f1_score(p, g)

hit1 /= len(test_a)
hit5 /= len(test_a)
f1 = f1_total / len(test_a)

# Simplified (single-answer ranking)
mrr = hit1
accuracy = hit1

print("===== LONGFORMER + NED + RERANKING =====")
print("Hit@1:", hit1)
print("Hit@5:", hit5)
print("MRR:", mrr)
print("F1:", f1)
print("Accuracy:", accuracy)